# Task 3 — Evaluable Users dan Leakage Rules

Seluruh model dievaluasi pada pengguna yang sama dan tidak menggunakan data validation/test untuk proses training.

Aturan yang diaudit:
- daftar evaluable users berasal dari `evaluable_users_validation_warm.csv`;
- seluruh evaluable users mempunyai histori pada training;
- validation hanya digunakan untuk membentuk target/evaluasi dan bukan untuk fitting model;
- test tidak digunakan untuk fitting model;
- seluruh model menerima user universe yang sama;
- Item-CF, TF-IDF CBF, dan Hybrid tidak membaca validation/test secara langsung;
- keberadaan Matrix Factorization diperiksa secara eksplisit.

## 1. Setup proyek

Jalankan notebook dari folder utama repository. Notebook mencari root secara otomatis agar tetap dapat dijalankan ketika current working directory berada di subfolder proyek.

In [1]:
from pathlib import Path
import ast
import inspect
import json
import re
import sys

import pandas as pd

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'data.py').is_file()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Root proyek tidak ditemukan. Jalankan notebook dari repository yang memiliki src/data.py.'
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_project_data
from src.evaluation import build_relevant_items, evaluate_ranking

print('Project root:', PROJECT_ROOT)

Project root: d:\Folder Kuliah\SEMESTER 7\SISREK\Project\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems


## 2. Muat data dan tetapkan sumber evaluasi

`load_project_data()` mengambil training, metadata, candidate universe warm, validation warm, dan daftar evaluable users resmi. Tidak ada split baru dan tidak ada pengacakan data.

In [2]:
data = load_project_data(PROJECT_ROOT)
train = data.train.copy()
items = data.items.copy()
validation_warm = data.validation_warm.copy()
candidates = list(dict.fromkeys(str(x) for x in data.candidates))
eval_users = list(dict.fromkeys(str(x) for x in data.evaluable_users))

for frame in (train, validation_warm):
    frame['user_id'] = frame['user_id'].astype(str)
    frame['item_id'] = frame['item_id'].astype(str)
items['item_id'] = items['item_id'].astype(str)

train['rating'] = pd.to_numeric(train['rating'], errors='raise')
validation_warm['rating'] = pd.to_numeric(validation_warm['rating'], errors='raise')

print(f'Training rows       : {len(train):,}')
print(f'Training users      : {train.user_id.nunique():,}')
print(f'Training items      : {train.item_id.nunique():,}')
print(f'Validation warm     : {len(validation_warm):,}')
print(f'Candidate items     : {len(candidates):,}')
print(f'Evaluable users     : {len(eval_users):,}')

assert len(train) == 427_957, 'Ukuran official training berbeda dari protokol proyek.'
assert eval_users, 'Daftar evaluable users kosong.'
assert candidates, 'Candidate universe kosong.'

Training rows       : 427,957
Training users      : 54,483
Training items      : 22,753
Validation warm     : 27,537
Candidate items     : 22,753
Evaluable users     : 11,659


## 3. Audit evaluable users

User evaluasi harus berasal dari file resmi dan harus sudah muncul pada training. Dengan demikian, protokol yang digunakan adalah warm-start, bukan cold-user evaluation.

In [3]:
train_users = set(train['user_id'])
eval_user_set = set(eval_users)
validation_users = set(validation_warm['user_id'])

missing_from_train = eval_user_set - train_users
validation_users_outside_eval = validation_users - eval_user_set

print('Evaluable users dari file resmi      :', len(eval_user_set))
print('Evaluable users yang ada di training :', len(eval_user_set & train_users))
print('User validation di luar evaluable   :', len(validation_users_outside_eval))

assert not missing_from_train, (
    f'{len(missing_from_train):,} evaluable users tidak memiliki histori training.'
)
assert not validation_users_outside_eval, (
    'Validation warm memuat user di luar daftar evaluable users.'
)

seen_by_user = {
    str(user): set(group['item_id'].astype(str))
    for user, group in train.groupby('user_id', sort=False)
}
print('Audit warm-start: LULUS')

Evaluable users dari file resmi      : 11659
Evaluable users yang ada di training : 11659
User validation di luar evaluable   : 0
Audit warm-start: LULUS


## 4. Audit target validation dan seen-item exclusion

Validation digunakan sebagai ground truth setelah model selesai dilatih. Item relevan memakai rating `>= 4`, berada di candidate universe, dan tidak boleh merupakan item yang sudah terlihat pada training user tersebut.

In [4]:
candidate_set = set(candidates)

relevant_rows = validation_warm.loc[
    validation_warm['user_id'].isin(eval_user_set)
    & validation_warm['item_id'].isin(candidate_set)
    & (validation_warm['rating'] >= 4)
].copy()

relevant_rows['training_seen'] = [
    item_id in seen_by_user.get(user_id, set())
    for user_id, item_id in zip(relevant_rows['user_id'], relevant_rows['item_id'])
]
n_seen_targets = int(relevant_rows['training_seen'].sum())
relevant_rows = relevant_rows.loc[~relevant_rows['training_seen']].copy()
relevant_by_user = (
    relevant_rows.groupby('user_id')['item_id'].apply(set).to_dict()
)
users_without_target = eval_user_set - set(relevant_by_user)

print('Target relevan sebelum seen-exclusion :', len(relevant_rows) + n_seen_targets)
print('Target validation yang training-seen  :', n_seen_targets)
print('Target relevan setelah filter         :', len(relevant_rows))
print('User tanpa target relevan             :', len(users_without_target))

assert n_seen_targets == 0, 'Ditemukan target validation yang sudah terlihat saat training.'
assert not users_without_target, 'Ada evaluable user tanpa target relevan pada protokol saat ini.'
assert all(targets <= candidate_set for targets in relevant_by_user.values())
print('Audit target validation: LULUS')

Target relevan sebelum seen-exclusion : 24202
Target validation yang training-seen  : 0
Target relevan setelah filter         : 24202
User tanpa target relevan             : 0
Audit target validation: LULUS


## 5. Semua model memakai user universe yang sama

Pada tahap rekomendasi, Popularity, Item-CF, TF-IDF CBF, dan Hybrid harus dipanggil dengan `eval_users` yang sama. Cell berikut mencatat kontrak user universe yang digunakan oleh eksperimen.

In [5]:
model_user_inputs = {
    'Popularity': eval_users,
    'Item-CF': eval_users,
    'TF-IDF CBF': eval_users,
    'Weighted Score Fusion': eval_users,
    'RRF': eval_users,
}

for model_name, users in model_user_inputs.items():
    assert set(map(str, users)) == eval_user_set, (
        f'{model_name} tidak memakai evaluable users yang sama.'
    )
    print(f'{model_name:22s}: {len(users):,} users')

# Smoke test: evaluator harus menghitung seluruh user yang memiliki target.
empty_recommendations = {user: [] for user in eval_users}
smoke_metrics = evaluate_ranking(empty_recommendations, relevant_by_user, eval_users, k=10)
print('Users evaluated oleh evaluator:', smoke_metrics['users_evaluated'])
assert smoke_metrics['users_evaluated'] == len(eval_users)
print('Audit user universe: LULUS')

Popularity            : 11,659 users
Item-CF               : 11,659 users
TF-IDF CBF            : 11,659 users
Weighted Score Fusion : 11,659 users
RRF                   : 11,659 users
Users evaluated oleh evaluator: 11659
Audit user universe: LULUS


## 6. Pemeriksaan statis terhadap kode model

Modul model dan fusion tidak boleh membaca file validation/test. Modul data boleh membaca validation karena tugasnya memang menyiapkan ground truth, tetapi fungsi `fit_*` hanya boleh menerima data training, metadata item, dan parameter model.

In [6]:
model_files = [PROJECT_ROOT / 'src' / 'recommenders.py', PROJECT_ROOT / 'src' / 'fusion.py']
forbidden_data_tokens = (
    'interactions_validation', 'validation_warm', 'interactions_test',
    'test_official', 'validation_official'
)

static_findings = []
for path in model_files:
    source = path.read_text(encoding='utf-8')
    hits = [token for token in forbidden_data_tokens if token in source]
    static_findings.append({
        'file': str(path.relative_to(PROJECT_ROOT)),
        'forbidden_validation_test_tokens': hits,
        'status': 'PASS' if not hits else 'REVIEW',
    })

display(pd.DataFrame(static_findings))
assert all(row['status'] == 'PASS' for row in static_findings), (
    'Modul model/fusion memiliki referensi langsung ke validation/test.'
)
print('Static leakage scan untuk src model/fusion: LULUS')

,file,forbidden_validation_test_tokens,status
0,src\recommenders.py,[],PASS
1,src\fusion.py,[],PASS


Static leakage scan untuk src model/fusion: LULUS


## 7. Pemeriksaan pemanggilan `fit_*` pada notebook

Pemeriksaan ini membaca code cell notebook secara statis. Pemanggilan fungsi fitting tidak boleh mengirim variabel `validation`, `validation_warm`, atau `test` sebagai argumen. Validation tetap boleh dipakai setelah fitting untuk evaluasi atau pemilihan parameter.

In [7]:
notebook_paths = [
    PROJECT_ROOT / 'Task_1_Experimental_Protocol.ipynb',
    PROJECT_ROOT / '02_baseline_recommenders.ipynb',
    PROJECT_ROOT / '03_hybrid_fusion.ipynb',
]

fit_calls = []
parse_warnings = []
for notebook_path in notebook_paths:
    if not notebook_path.exists():
        continue
    notebook = json.loads(notebook_path.read_text(encoding='utf-8'))
    for cell_number, cell in enumerate(notebook.get('cells', [])):
        if cell.get('cell_type') != 'code':
            continue
        source = ''.join(cell.get('source', []))
        try:
            tree = ast.parse(source)
        except SyntaxError as error:
            parse_warnings.append(f'{notebook_path.name} cell {cell_number}: {error.msg}')
            continue
        for node in ast.walk(tree):
            if not isinstance(node, ast.Call):
                continue
            function_name = (
                node.func.id if isinstance(node.func, ast.Name)
                else node.func.attr if isinstance(node.func, ast.Attribute)
                else ''
            )
            if function_name.startswith('fit_'):
                call_text = ast.unparse(node)
                fit_calls.append({
                    'notebook': notebook_path.name,
                    'cell': cell_number,
                    'call': call_text,
                    'contains_validation_or_test': bool(
                        re.search(r'\b(validation|validation_warm|test|test_official)\b', call_text)
                    ),
                })

fit_calls_df = pd.DataFrame(fit_calls)
display(fit_calls_df)
if parse_warnings:
    print('Peringatan parsing:', *parse_warnings, sep='\n- ')

assert not fit_calls_df.empty, 'Tidak ditemukan pemanggilan fit_* pada notebook model.'
assert not fit_calls_df['contains_validation_or_test'].any(), (
    'Ada pemanggilan fit_* yang mengirim validation/test sebagai argumen.'
)
print('Notebook fit-call leakage scan: LULUS')

,notebook,cell,call,contains_validation_or_test
0,Task_1_Experimental_Protocol.ipynb,15,"fit_popularity(train, positive_threshold=POSIT...",False
1,Task_1_Experimental_Protocol.ipynb,18,"fit_item_cf(train, candidates, top_k=100, posi...",False
2,Task_1_Experimental_Protocol.ipynb,21,vectorizer.fit_transform(documents),False
3,02_baseline_recommenders.ipynb,3,"fit_popularity(train, positive_threshold=4)",False
4,02_baseline_recommenders.ipynb,7,"fit_item_cf(train, candidates, top_k=100, posi...",False
5,02_baseline_recommenders.ipynb,11,"fit_tfidf_cbf(train, items, candidates, text_c...",False
6,03_hybrid_fusion.ipynb,1,"fit_popularity(train, positive_threshold=4)",False
7,03_hybrid_fusion.ipynb,3,"fit_item_cf(train, candidates, top_k=100, posi...",False
8,03_hybrid_fusion.ipynb,3,"fit_tfidf_cbf(train, items, candidates, text_c...",False


Notebook fit-call leakage scan: LULUS


## 8. Pemeriksaan Matrix Factorization

Task 3 mencakup MF, tetapi repository saat ini berisi Popularity, Item-CF, dan TF-IDF CBF. Cell berikut tidak menganggap Item-CF sebagai Matrix Factorization dan hanya melaporkan apakah implementasi MF benar-benar tersedia.

In [8]:
mf_pattern = re.compile(r'(matrix[_ -]?factor|\bmf\b|fit_mf|\bsvd\b|\bals\b)', re.IGNORECASE)
mf_files = []
mf_hits = []
for path in (PROJECT_ROOT / 'src').rglob('*.py'):
    source = path.read_text(encoding='utf-8')
    if mf_pattern.search(path.name) or mf_pattern.search(source):
        mf_files.append(str(path.relative_to(PROJECT_ROOT)))
        mf_hits.append(str(path.relative_to(PROJECT_ROOT)))

if mf_files:
    print('Implementasi MF terdeteksi:', mf_files)
    mf_status = 'FOUND - perlu audit lanjutan'
else:
    print('Implementasi MF tidak ditemukan di src/.')
    mf_status = 'NOT_AVAILABLE - belum ada implementasi MF pada repository saat ini'

print('Status MF:', mf_status)

Implementasi MF tidak ditemukan di src/.
Status MF: NOT_AVAILABLE - belum ada implementasi MF pada repository saat ini


## 9. Audit test set dan ringkasan hasil

Test set tidak tersedia di paket proyek saat ini. Yang penting untuk leakage rule adalah tidak ada fungsi fitting yang membaca validation/test dan tidak ada data masa depan yang masuk ke training. Hasil audit disimpan agar dapat dijadikan bukti Task 3.

In [9]:
main_dirs = list(PROJECT_ROOT.glob('MAIN DATA-*/MAIN DATA'))
test_files = sorted(main_dirs[0].glob('interactions_test*.csv')) if len(main_dirs) == 1 else []

audit_rows = [
    {'check': 'Official evaluable users digunakan', 'status': 'PASS', 'evidence': f'{len(eval_users):,} users'},
    {'check': 'Semua evaluable users ada di training', 'status': 'PASS', 'evidence': f'{len(missing_from_train):,} missing'},
    {'check': 'Validation user berada di evaluable universe', 'status': 'PASS', 'evidence': f'{len(validation_users_outside_eval):,} outside'},
    {'check': 'Seen-item exclusion pada target', 'status': 'PASS', 'evidence': f'{n_seen_targets:,} overlap'},
    {'check': 'Semua model memakai user universe sama', 'status': 'PASS', 'evidence': f'{len(eval_users):,} users/model'},
    {'check': 'src model/fusion bebas dari validation/test read', 'status': 'PASS', 'evidence': 'static scan'},
    {'check': 'Notebook fit-call bebas dari validation/test args', 'status': 'PASS', 'evidence': f'{len(fit_calls_df):,} fit calls'},
    {'check': 'Test file tersedia', 'status': 'INFO' if test_files else 'NOT_AVAILABLE', 'evidence': [str(p) for p in test_files] if test_files else 'tidak tersedia'},
    {'check': 'Matrix Factorization', 'status': 'INFO', 'evidence': mf_status},
]

audit_df = pd.DataFrame(audit_rows)
display(audit_df)

output_dir = PROJECT_ROOT / 'outputs' / 'task3'
output_dir.mkdir(parents=True, exist_ok=True)
summary = {
    'task': 'Task 3 - Evaluable Users and Leakage Rules',
    'train_interactions': int(len(train)),
    'evaluable_users': int(len(eval_users)),
    'validation_warm_interactions': int(len(validation_warm)),
    'target_rows_after_seen_exclusion': int(len(relevant_rows)),
    'users_without_target': int(len(users_without_target)),
    'test_files_available': [str(p.relative_to(PROJECT_ROOT)) for p in test_files],
    'mf_status': mf_status,
    'fit_call_count_audited': int(len(fit_calls_df)),
    'leakage_checks': 'PASS',
}
(output_dir / 'task3_audit_summary.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
audit_df.to_csv(output_dir / 'task3_leakage_audit.csv', index=False)

print('Task 3 selesai.')
print('-', output_dir / 'task3_audit_summary.json')
print('-', output_dir / 'task3_leakage_audit.csv')

,check,status,evidence
0,Official evaluable users digunakan,PASS,"11,659 users"
1,Semua evaluable users ada di training,PASS,0 missing
2,Validation user berada di evaluable universe,PASS,0 outside
3,Seen-item exclusion pada target,PASS,0 overlap
4,Semua model memakai user universe sama,PASS,"11,659 users/model"
5,src model/fusion bebas dari validation/test read,PASS,static scan
6,Notebook fit-call bebas dari validation/test args,PASS,9 fit calls
7,Test file tersedia,NOT_AVAILABLE,tidak tersedia
8,Matrix Factorization,INFO,NOT_AVAILABLE - belum ada implementasi MF pada...


Task 3 selesai.
- d:\Folder Kuliah\SEMESTER 7\SISREK\Project\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems\outputs\task3\task3_audit_summary.json
- d:\Folder Kuliah\SEMESTER 7\SISREK\Project\Hybrid-Fusion-Strategies-for-Collaborative-and-Content-Based-Recommender-Systems\outputs\task3\task3_leakage_audit.csv


## Kesimpulan

Pada repository saat ini, evaluable users berasal dari file resmi, seluruhnya mempunyai histori training, dan kode model/fusion tidak membaca validation atau test untuk fitting. Matrix Factorization belum tersedia sehingga belum dapat diaudit atau dibandingkan pada Task 3; Item-CF tidak dikategorikan sebagai MF.